# Setup and GPU benchmark

Uses synthetic tensors only, so its output is not a training result. Share `benchmark.json` from Drive when finished.

In [ ]:
# Run this cell at the start of every Colab session. Runtime type must be GPU.
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/FastSpeech2-IIITH'
REPO = '/content/FastSpeech2-IIITH'
REPO_URL = ''  # optional: GitHub URL once the repository is public
os.makedirs(DRIVE, exist_ok=True)
if not os.path.isdir(REPO):
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
    else:
        subprocess.run(['unzip', '-q', f'{DRIVE}/FastSpeech2-IIITH-code.zip', '-d', REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, REPO)
from src.data.storage import configure_environment
# HF_HOME is deliberately left alone, so Hugging Face's token file stays off Drive.
configure_environment(DRIVE)
os.environ.update(FS2_REPO_ROOT=REPO, PYTHONUNBUFFERED='1')
!nvidia-smi
import torch
print('PyTorch', torch.__version__, 'CUDA build', torch.version.cuda, 'GPU available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU'


## Hugging Face access

Confirms the `HF_TOKEN` secret works. The token is never printed or saved to Drive.

In [ ]:
# Add a Read token first: key icon in the left sidebar > Add new secret > name HF_TOKEN > allow this notebook.
from src.data.hf_auth import authenticate
authenticate()


In [ ]:
!python -m pytest -q tests

In [ ]:
!python -m src.training.benchmark --out "$FS2_EXP_ROOT/benchmark.json" --steps 30
!cat "$FS2_EXP_ROOT/benchmark.json"